# businessDX on Google Colab

この Notebook は Google Colab 上で monthly Excel / PDF を処理し、出力を Google Drive に保存するための実行入口です。

## 1. リポジトリ取得

`REPO_URL` を必要に応じて変更してください。すでに `/content/businessDX_unyou3` に配置済みなら clone 行は不要です。

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/yunhuu61/businessDX_unyou3.git'
REPO_DIR = Path('/content/businessDX_unyou3')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repository found: {REPO_DIR}')

## 2. 依存ライブラリのインストール

In [ ]:
%cd /content/businessDX_unyou3
!pip install -q -r src/requirements.txt

## 3. Google Drive をマウント

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

## 4. ライブラリ読み込み

In [ ]:
import getpass
import shutil
import sys
from pathlib import Path

sys.path.append('/content/businessDX_unyou3/src')

from app_config import AppConfig
from pipeline import run_pipeline

## 5. 月次ファイルをアップロード

Excel と PDF をこのセルでアップロードします。

In [ ]:
uploaded = files.upload()

## 6. 実行設定を作成

In [ ]:
input_dir = Path('/content/businessdx_input')
if input_dir.exists():
    shutil.rmtree(input_dir)
input_dir.mkdir(parents=True, exist_ok=True)

for filename, content in uploaded.items():
    destination = input_dir / filename
    destination.write_bytes(content)

excel_files = sorted(path.name for path in input_dir.glob('*.xlsx') if not path.name.startswith('~$'))
pdf_files = sorted(path.name for path in input_dir.glob('*.pdf'))
print('Excel files:', excel_files)
print('PDF files:', pdf_files)

gemini_api_key = getpass.getpass('Gemini API key: ').strip()
run_yyyymm = input('Run yyyymm (example: 202603): ').strip()
run_pdf = input('Run PDF split? [y/n]: ').strip().lower() != 'n'
run_excel = input('Run Excel analysis? [y/n]: ').strip().lower() != 'n'

classification_csv_path = Path('/content/drive/MyDrive/businessDX/setting/分類.csv')
output_dir = Path(f'/content/drive/MyDrive/businessDX/Output/{run_yyyymm}')

if not run_yyyymm:
    raise ValueError('yyyyMM is required.')
if run_excel and not gemini_api_key:
    raise ValueError('Gemini API key is required when Excel analysis is enabled.')
if run_excel and not classification_csv_path.exists():
    raise FileNotFoundError(f'Classification CSV not found: {classification_csv_path}')
if run_excel and not excel_files:
    raise FileNotFoundError('No Excel files were uploaded.')
if run_pdf and not pdf_files:
    raise FileNotFoundError('No PDF files were uploaded.')

config = AppConfig(
    input_dir=str(input_dir),
    output_dir=str(output_dir),
    classification_csv_path=str(classification_csv_path),
    gemini_api_key=gemini_api_key,
    run_pdf=run_pdf,
    run_excel=run_excel,
)

print('Output dir:', output_dir)
print('Classification CSV:', classification_csv_path)
print('Enabled steps:', config.enabled_steps())

## 7. 実行

In [ ]:
result = run_pipeline(config)
result

## 8. 出力確認

In [ ]:
print('Saved to:', output_dir)
for path in sorted(output_dir.rglob('*')):
    print(path)